# 使用args_schema

In [3]:
import os
from tempfile import template

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langchain_openai import ChatOpenAI
from traitlets.utils import descriptions

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
llm_deepseek = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)

In [11]:

from langchain_core.tools.convert import tool
from pydantic import BaseModel, Field
from langchain_core.utils.function_calling import convert_to_openai_tool


# class WeatherSchema(BaseModel):
#     city:str = Field(default="北京" , description="城市")
#     is_forecast:bool= Field(default=True , description="是否进行预测")
#
#
# @tool("get_weather_and_forecast" , description="查询今天天气,可以包含明天预测天气" , args_schema=WeatherSchema)
# def get_weather_and_forecast(city:str , is_forecast:bool):
#     res = f"{city}今天天气不错"
#     if is_forecast:
#         res += "明天下雨"
#     return res


# =============================
@tool("get_weather_and_forecast" ,parse_docstring=True)
def get_weather_and_forecast(city:str , is_forecast:bool):
    """
    "查询今天天气,可以包含明天预测天气

    Args:
        city : 查询的城市
        is_forecast : 是否进行预测
    """
    res = f"{city}今天天气不错"
    if is_forecast:
        res += "明天下雨"
    return res

print(convert_to_openai_tool(get_weather_and_forecast))


{'type': 'function', 'function': {'name': 'get_weather_and_forecast', 'description': '"查询今天天气,可以包含明天预测天气', 'parameters': {'properties': {'city': {'description': '查询的城市', 'type': 'string'}, 'is_forecast': {'description': '是否进行预测', 'type': 'boolean'}}, 'required': ['city', 'is_forecast'], 'type': 'object'}}}


In [7]:
from langchain_core.messages import HumanMessage , AIMessage

model_with_tools = llm_deepseek.bind_tools([get_weather_and_forecast])

# 维护消息列表
messages = [HumanMessage("今天西安天气怎么样,明天呢")]

response = model_with_tools.invoke(messages)

messages.append(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
        tool_res = get_weather_and_forecast.invoke(tool_call)
        messages.append(tool_res)

fin_res = model_with_tools.invoke(messages)

messages.append(fin_res)

for message in messages:
    message.pretty_print()



================================ Human Message =================================

今天西安天气怎么样,明天呢
================================== Ai Message ==================================
Tool Calls:
  get_weather_and_forecast (call_00_RvE3i8COVmuFI1GMNplo0723)
 Call ID: call_00_RvE3i8COVmuFI1GMNplo0723
  Args:
    city: 西安
    is_forecast: True
================================= Tool Message =================================
Name: get_weather_and_forecast

西安今天天气不错明天下雨
================================== Ai Message ==================================

西安今天天气不错，明天有雨。建议您今天可以放心安排户外活动，明天出门记得带伞哦！
